<a href="https://colab.research.google.com/github/kang25-gif/BA810/blob/main/Lab9_SVM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 9: Support Vector Machines

In this lab we we apply Support Vector Machines to predict which brand of orange juice a customer will buy. We'll use the [`OJ: Orange Juice Data`](https://www.rdocumentation.org/packages/ISLR2/versions/1.3-1/topics/OJ). You can get the data file from [our course data folder](https://drive.google.com/drive/folders/1YBxtx7KILtcw6mFEkKmznaRASUHPLjI2?usp=sharing).

The data set has the following columns:

1. `Purchase` (the **target**): A categorical variable with levels CH and MM indicating whether the customer purchased Citrus Hill or Minute Maid Orange Juice
1. `WeekofPurchase`: Week of purchase
1. `StoreID`: Store ID
1. `PriceCH`: Price charged for CH
1. `PriceMM`: Price charged for MM
1. `DiscCH`: Discount offered for CH
1. `DiscMM`: Discount offered for MM
1. `SpecialCH`: Indicator of special on CH
1. `SpecialMM`: Indicator of special on MM
1. `LoyalCH`: Customer brand loyalty for CH
1. `SalePriceMM`: Sale price for MM
1. `SalePriceCH`: Sale price for CH
1. `PriceDiff`: Sale price of MM less sale price of CH
1. `Store7`: A categorical variable with levels No and Yes indicating whether the sale is at Store 7
1. `PctDiscMM`: Percentage discount for MM
1. `PctDiscCH`: Percentage discount for CH
1. `ListPriceDiff`: List price of MM less list price of CH
1. `STORE`: Which of 5 possible stores the sale occured at. It codes some store ids using different numbers, e.g. 7 → 0.

The broad outline of the lab is as follows:

1. Explore, clean, and split the dataset
1. Train and examine a Support Vector Classifier
1. Evaluate various SVM kernels and search for the best hypter-parameter values using cross validation
1. Test the best model on the test data



In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# drive_url = 'https://drive.google.com/file/d/1wECUewbW2-OKfsaBo4Atydv-2H7J2kHK/view?usp=drive_link'
file_url = 'https://docs.google.com/uc?export=download&id=1wECUewbW2-OKfsaBo4Atydv-2H7J2kHK'

rawdata = pd.read_csv(file_url)
rawdata.head(10)

,Purchase,WeekofPurchase,StoreID,PriceCH,PriceMM,DiscCH,DiscMM,SpecialCH,SpecialMM,LoyalCH,SalePriceMM,SalePriceCH,PriceDiff,Store7,PctDiscMM,PctDiscCH,ListPriceDiff,STORE
0,CH,237,1,1.75,1.99,0.00,0.0,0,0,0.500000,1.99,1.75,0.24,No,0.000000,0.000000,0.24,1
1,CH,239,1,1.75,1.99,0.00,0.3,0,1,0.600000,1.69,1.75,-0.06,No,0.150754,0.000000,0.24,1
2,CH,245,1,1.86,2.09,0.17,0.0,0,0,0.680000,2.09,1.69,0.40,No,0.000000,0.091398,0.23,1
3,MM,227,1,1.69,1.69,0.00,0.0,0,0,0.400000,1.69,1.69,0.00,No,0.000000,0.000000,0.00,1
4,CH,228,7,1.69,1.69,0.00,0.0,0,0,0.956535,1.69,1.69,0.00,Yes,0.000000,0.000000,0.00,0
5,CH,230,7,1.69,1.99,0.00,0.0,0,1,0.965228,1.99,1.69,0.30,Yes,0.000000,0.000000,0.30,0
6,CH,232,7,1.69,1.99,0.00,0.4,1,1,0.972182,1.59,1.69,-0.10,Yes,0.201005,0.000000,0.30,0
7,CH,234,7,1.75,1.99,0.00,0.4,1,0,0.977746,1.59,1.75,-0.16,Yes,0.201005,0.000000,0.24,0
8,CH,235,7,1.75,1.99,0.00,0.4,0,0,0.982197,1.59,1.75,-0.16,Yes,0.201005,0.000000,0.24,0
9,CH,238,7,1.75,1.99,0.00,0.4,0,0,0.985757,1.59,1.75,-0.16,Yes,0.201005,0.000000,0.24,0


## Load, explore, and prepare dataset

First make sure that you have got the `OJ.csv` file from the linked drive and placed it in the appropriate folder. Then mount the google drive and read the file.

In [2]:
rawdata.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1070 entries, 0 to 1069
Data columns (total 18 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Purchase        1070 non-null   object 
 1   WeekofPurchase  1070 non-null   int64  
 2   StoreID         1070 non-null   int64  
 3   PriceCH         1070 non-null   float64
 4   PriceMM         1070 non-null   float64
 5   DiscCH          1070 non-null   float64
 6   DiscMM          1070 non-null   float64
 7   SpecialCH       1070 non-null   int64  
 8   SpecialMM       1070 non-null   int64  
 9   LoyalCH         1070 non-null   float64
 10  SalePriceMM     1070 non-null   float64
 11  SalePriceCH     1070 non-null   float64
 12  PriceDiff       1070 non-null   float64
 13  Store7          1070 non-null   object 
 14  PctDiscMM       1070 non-null   float64
 15  PctDiscCH       1070 non-null   float64
 16  ListPriceDiff   1070 non-null   float64
 17  STORE           1070 non-null   i

There don't seem to be any missing values: all columns have 1070 non-null values, in 1070 records. There are a few other things that require our attention though:

1. `StoreID`, `SpecialCH`, and `SpecialMM` should be categorical variables. Though `Purchase` can be left as object, storing as category is more efficient.
1. `Store7` and `STORE` contain data that same or coarser versions of `StoreID`. We can drop these two derived columns.
1. `PriceDiff` and `ListPriceDiff` are derevied from other price columns too; they can be dropped or kept as engineered features.

Let's make these changes first.

In [3]:
data = rawdata.astype({'Purchase': 'category', 'StoreID':'category', 'SpecialCH':'category', 'SpecialMM':'category', })
data.drop(['Store7', 'STORE'], axis=1, inplace=True)
data.info()
data.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1070 entries, 0 to 1069
Data columns (total 16 columns):
 #   Column          Non-Null Count  Dtype   
---  ------          --------------  -----   
 0   Purchase        1070 non-null   category
 1   WeekofPurchase  1070 non-null   int64   
 2   StoreID         1070 non-null   category
 3   PriceCH         1070 non-null   float64 
 4   PriceMM         1070 non-null   float64 
 5   DiscCH          1070 non-null   float64 
 6   DiscMM          1070 non-null   float64 
 7   SpecialCH       1070 non-null   category
 8   SpecialMM       1070 non-null   category
 9   LoyalCH         1070 non-null   float64 
 10  SalePriceMM     1070 non-null   float64 
 11  SalePriceCH     1070 non-null   float64 
 12  PriceDiff       1070 non-null   float64 
 13  PctDiscMM       1070 non-null   float64 
 14  PctDiscCH       1070 non-null   float64 
 15  ListPriceDiff   1070 non-null   float64 
dtypes: category(4), float64(11), int64(1)
memory usage: 105.2 KB

,Purchase,WeekofPurchase,StoreID,PriceCH,PriceMM,DiscCH,DiscMM,SpecialCH,SpecialMM,LoyalCH,SalePriceMM,SalePriceCH,PriceDiff,PctDiscMM,PctDiscCH,ListPriceDiff
0,CH,237,1,1.75,1.99,0.00,0.0,0,0,0.500000,1.99,1.75,0.24,0.000000,0.000000,0.24
1,CH,239,1,1.75,1.99,0.00,0.3,0,1,0.600000,1.69,1.75,-0.06,0.150754,0.000000,0.24
2,CH,245,1,1.86,2.09,0.17,0.0,0,0,0.680000,2.09,1.69,0.40,0.000000,0.091398,0.23
3,MM,227,1,1.69,1.69,0.00,0.0,0,0,0.400000,1.69,1.69,0.00,0.000000,0.000000,0.00
4,CH,228,7,1.69,1.69,0.00,0.0,0,0,0.956535,1.69,1.69,0.00,0.000000,0.000000,0.00


Let's examine the outcome distribution for any significant class imbalance.

In [4]:
data['Purchase'].value_counts(normalize=True)

,proportion
Purchase,
CH,0.61028
MM,0.38972


There doesn't seem to be any. We can use `accuracy` to measure performances.

In a typical ML project we should examine the histograms and scatters to understand the data a little better. We'll skip it in this lab to stay focused on SVM, but you should do that.


Next, separate the `X` from `y`, then split all into training and testing sets.

In [5]:
from sklearn.model_selection import train_test_split
X = data.drop('Purchase', axis=1) # separate X ...
y = data['Purchase'].copy()       # from y
train_X, test_X, train_y, test_y = train_test_split(X, y, test_size = .2, random_state=0) #split all
train_X.shape, test_X.shape, train_y.shape, test_y.shape # check sizes

((856, 15), (214, 15), (856,), (214,))

## Train and examine an SVM

We should create the standard preprocessing pipeline that we have seen in the previous labs: potential imputation followed by standardization for numeric variables and OneHotEncoding for categorical variables.

SVMs are sensitve to scales of the variables: like k-nearest neighbor classifiers these are distance based too. The features with larger magnitude and variance will dominate distance calculation. So, they don't work well when variables are in very different scales. Hence, standardization is important.

In [17]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer, make_column_selector

from sklearn import set_config
set_config(display='diagram') # shows the pipeline graphically when printed

num_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ])
cat_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('cat_encoder', OneHotEncoder())
    ])

prep_pipeline = ColumnTransformer([
    ('num', num_pipeline, make_column_selector(dtype_include=np.number)),
    ('cat', cat_pipeline, make_column_selector(dtype_include='category'))
])

prep_pipeline

ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 <sklearn.compose._column_transformer.make_column_selector object at 0x785e742181a0>),
                                ('cat',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('cat_encoder',
                                                  OneHotEncoder())]),
                                 <sklearn.compose._column_transformer.make_column_selector object at 0x785e751f0440>)])

In the following block we examine the attributes of the trained SVM (e.g., the support vectors). The data we get are as numpy arrays, not pandas data frames.

In [18]:
# @title Check support vectors {run: "auto"}

from sklearn.svm import SVC

c = 10 # @param {type:"slider", min:1, max:10}

lin_svc = SVC(C=c, kernel='poly', degree=1)

svm_pipeline = Pipeline([
    ("preprocessing", prep_pipeline),
    ("svm", lin_svc),
])

svm_pipeline.fit(train_X, train_y)

# Print the number of support vectors for each class

print('Number of support vectors for each class:',
      dict(zip(lin_svc.classes_, lin_svc.n_support_)))
# See how number of support vectors change as you increase C.

# You can also print the records that are the support vectors.
# print(lin_svc.support_vectors_)

Number of support vectors for each class: {'CH': np.int32(177), 'MM': np.int32(174)}


What happens to the number of support vectors when you increase/decrease $C$? Why?

Now let's check the accuracy of the predictions on test data.

In [24]:
from sklearn.metrics import accuracy_score

print(f'Test accuracy was {accuracy_score(test_y, svm_pipeline.predict(test_X)):.2f}' )

NotFittedError: Pipeline is not fitted yet.

## Tune SVMs

Various kernels in SVMs require tuning to perform well. We can use one of the search strategies we have learnt to do that. Let's start with the grid search.

In [19]:
from sklearn.model_selection import GridSearchCV

svm_pipeline = Pipeline([
    ("preprocessing", prep_pipeline),
    ("svm", SVC()),
])

param_grid = [
  {'svm__kernel': ['linear'], 'svm__C': [1, 10, 100, 1000]},
  {'svm__kernel': ['rbf'], 'svm__C': [1, 10, 100, 1000], 'svm__gamma': [0.001, 0.0001]},
  {'svm__kernel': ['poly'], 'svm__C': [1, 10, 100, 1000], 'svm__gamma': [0.001, 0.0001], 'svm__degree': [2, 3, 4]},
]
# Notice the list of dictionaries syntax: it allows us to explore a different set of parameters for each kernel.
# The grid search explores dictionaries sequentially. For each dictionary it evaluates all hyper-parameter combinations.
# Random search allows something similar too — instead of list of hyperparameter values, it takes distributions.
# With random search the list of dictionary is sampled uniformly first for each iteration, then the
# hyper-parameters within it from their specified distributions. BayesSearchCV on the other hand
# draws n_iter samples for **each** dictionary.

grid_search = GridSearchCV(svm_pipeline, param_grid, cv=3, scoring='accuracy')
grid_search.fit(train_X, train_y)

grid_cv_res = pd.DataFrame(grid_search.cv_results_)
grid_cv_res.sort_values(by="mean_test_score", ascending=False, inplace=True)
grid_cv_res.filter(regex = '(^param_|mean_test_score)', axis=1)

,param_svm__C,param_svm__kernel,param_svm__gamma,param_svm__degree,mean_test_score
0,1,linear,NaN,NaN,0.836474
10,1000,rbf,0.0010,NaN,0.836474
8,100,rbf,0.0010,NaN,0.836474
1,10,linear,NaN,NaN,0.834139
11,1000,rbf,0.0001,NaN,0.834139
2,100,linear,NaN,NaN,0.831804
3,1000,linear,NaN,NaN,0.830634
6,10,rbf,0.0010,NaN,0.830630
9,100,rbf,0.0001,NaN,0.829461
30,1000,poly,0.0010,2.0,0.789752


As the conventional wisdom would suggest, the linear SVM seems to be working well in this setting along with RBF. There is a separate classifier called `LinearSVC` that contains an optimized implementation of the linear SVM. Polynomial kernels don't work as well as the others in this setting.

(Why do we have NaNs in the above results? Because some hyperparameters aren't applicable/present for some kernels, e.g., `gamma` isn't present for linear kernel.)

In [26]:
# We'll work with the best model obtained from grid search.
model = grid_search.best_estimator_

# What is the accuracy if we applied that to the test data we set aside at near the beginning?
pred_y = model.predict(test_X)
print('The cost under standard prediction strategy is %.2f.' % accuracy_score(test_y, pred_y))

The cost under standard prediction strategy is 0.82.


***
**Exercise**

SVMs have many hyper-parameters that can take values in a wide range. Besides, the performance of the SVMs depends quite a bit on choice of right hyper-parameter values. To make matters even more interesting, different kernels take different hyper-parameters. Thus, SVM application is a prime candidate for randomized search.

1. Apply `RandomizedSearchCV` to select parameters for our classification exercise. Can you get a better classifier than what grid search found?
1. Then use `HalvingRandomSearchCV` to further increase the exploration. Use `loguniform` distribution for `C` and `gamma`, and `randint` for `degree`.
1. We also learnt about intelligently searching for hyper-prameter values using bayesian search. Use the `BayesSearchCV` from `scikit-optimize` to see if we can get an even better model. See [scikit optimize page](https://scikit-optimize.github.io/stable/modules/space.html) for how to specify loguniform distribution.

***


In [29]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import loguniform, randint
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline

# Ensure the prep_pipeline is available (it should be from previous cells)
# svm_pipeline_rand is a new pipeline instance for RandomizedSearchCV
svm_pipeline_rand = Pipeline([
    ("preprocessing", prep_pipeline),
    ("svm", SVC()),
])

# Define the parameter distributions for RandomizedSearchCV
# Using loguniform for C and gamma, and randint for degree as specified in the exercise
param_distributions = [
  {'svm__kernel': ['linear'], 'svm__C': loguniform(1, 1000)},
  {'svm__kernel': ['rbf'], 'svm__C': loguniform(1, 1000), 'svm__gamma': loguniform(0.0001, 1)},
  {'svm__kernel': ['poly'], 'svm__C': loguniform(1, 1000), 'svm__gamma': loguniform(0.0001, 1), 'svm__degree': randint(2, 5)},
]

# Instantiate RandomizedSearchCV
random_search = RandomizedSearchCV(
    svm_pipeline_rand,
    param_distributions,
    n_iter=50, # Number of parameter settings that are sampled
    cv=3,
    scoring='accuracy',
    random_state=42, # for reproducibility
    n_jobs=-1 # Use all available cores
)

# Fit RandomizedSearchCV to the training data
random_search.fit(train_X, train_y)

# Print the best parameters and best score
print("Best parameters from RandomizedSearchCV:", random_search.best_params_)
print("Best cross-validation accuracy from RandomizedSearchCV: %.4f" % random_search.best_score_)

# Evaluate on test data
print(f'Test accuracy with best RandomizedSearchCV model: {random_search.best_estimator_.score(test_X, test_y):.2f}')


Best parameters from RandomizedSearchCV: {'svm__C': np.float64(14.656553886225328), 'svm__kernel': 'linear'}
Best cross-validation accuracy from RandomizedSearchCV: 0.8365
Test accuracy with best RandomizedSearchCV model: 0.81


In [39]:
rand_cv_res = pd.DataFrame(random_search.cv_results_)
rand_cv_res.sort_values(by="mean_test_score", ascending=False, inplace=True)
rand_cv_res.filter(regex = '(^param_|mean_test_score)', axis=1)

,param_svm__C,param_svm__degree,param_svm__gamma,param_svm__kernel,mean_test_score
26,2.647114,NaN,NaN,linear,0.836478
23,14.656554,NaN,NaN,linear,0.836478
46,4.828425,NaN,NaN,linear,0.836478
11,1.378324,NaN,NaN,linear,0.836474
44,446.519857,NaN,0.001983,rbf,0.836470
28,3.945909,NaN,NaN,linear,0.835309
8,37.525283,NaN,NaN,linear,0.835309
17,5.975028,NaN,NaN,linear,0.834139
7,37.520559,NaN,0.005343,rbf,0.834135
1,61.737704,NaN,NaN,linear,0.834135


In [34]:
from sklearn.experimental import enable_halving_search_cv # noqa
from sklearn.model_selection import HalvingRandomSearchCV
from scipy.stats import loguniform, randint
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score

# Ensure the prep_pipeline is available from previous cells
# svm_pipeline_halving is a new pipeline instance for HalvingRandomSearchCV
svm_pipeline_halving = Pipeline([
    ("preprocessing", prep_pipeline),
    ("svm", SVC()),
])

# Define the parameter distributions for HalvingRandomSearchCV
param_distributions_halving = [
  {'svm__kernel': ['linear'], 'svm__C': loguniform(1, 1000)},
  {'svm__kernel': ['rbf'], 'svm__C': loguniform(1, 1000), 'svm__gamma': loguniform(0.0001, 1)},
  {'svm__kernel': ['poly'], 'svm__C': loguniform(1, 1000), 'svm__gamma': loguniform(0.0001, 1), 'svm__degree': randint(2, 5)},
]

# Instantiate HalvingRandomSearchCV
# Using 'n_samples' as the resource, default max_resources is the number of samples in the training set
halving_random_search = HalvingRandomSearchCV(
    svm_pipeline_halving,
    param_distributions_halving,
    factor=3, # As suggested in the instructions
    cv=3,
    scoring='accuracy',
    random_state=42,
    n_jobs=-1,
    resource='n_samples' # Explicitly set resource as per instruction
)

# Fit HalvingRandomSearchCV to the training data
halving_random_search.fit(train_X, train_y)

# Print the best parameters and best score
print("Best parameters from HalvingRandomSearchCV:", halving_random_search.best_params_)
print("Best cross-validation accuracy from HalvingRandomSearchCV: %.4f" % halving_random_search.best_score_)

# Evaluate on test data
print(f'Test accuracy with best HalvingRandomSearchCV model: {halving_random_search.best_estimator_.score(test_X, test_y):.2f}')


Best parameters from HalvingRandomSearchCV: {'svm__C': np.float64(37.52055855124281), 'svm__gamma': np.float64(0.005342937261279773), 'svm__kernel': 'rbf'}
Best cross-validation accuracy from HalvingRandomSearchCV: 0.7856
Test accuracy with best HalvingRandomSearchCV model: 0.81


In [40]:
halv_cv_res = pd.DataFrame(halving_random_search.cv_results_)
halv_cv_res.sort_values(by="mean_test_score", ascending=False, inplace=True)
halv_cv_res.filter(regex = '(^param_|mean_test_score)', axis=1)

,param_svm__C,param_svm__degree,param_svm__gamma,param_svm__kernel,mean_test_score
105,37.520559,NaN,0.005343,rbf,0.785595
101,37.520559,NaN,0.005343,rbf,0.782540
103,9.452571,NaN,0.082875,rbf,0.779422
104,41.598378,NaN,0.022233,rbf,0.776278
96,41.598378,NaN,0.022233,rbf,0.763757
...,...,...,...,...,...
66,4.578499,3.0,0.011178,poly,NaN
67,11.117265,4.0,0.000498,poly,NaN
68,15.585332,NaN,NaN,linear,NaN
69,645.936709,2.0,0.050438,poly,NaN


In [ ]:
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

# Import BayesSearchCV and space objects
from skopt import BayesSearchCV
from skopt.space import Real, Integer, Categorical

# Create a new pipeline instance for BayesSearchCV
svm_pipeline_bayes = Pipeline([
    ("preprocessing", prep_pipeline),
    ("svm", SVC()),
])

# Define the parameter space for BayesSearchCV
# Note: BayesSearchCV takes a single dictionary for the search space, not a list of dictionaries like GridSearchCV/RandomizedSearchCV
# We need to explicitly define the categorical kernel and then specify parameters conditionally or define a broader space.
# For simplicity and to match the previous approach, we'll define a single space that covers all kernels,
# but only relevant parameters will be used based on the 'svm__kernel' choice by SVC internally.

search_space = [
    {
        'svm__kernel': Categorical(['linear']),
        'svm__C': Real(1e-1, 1e3, prior='log-uniform')
    },
    {
        'svm__kernel': Categorical(['rbf']),
        'svm__C': Real(1e-1, 1e3, prior='log-uniform'),
        'svm__gamma': Real(1e-4, 1e0, prior='log-uniform')
    },
    {
        'svm__kernel': Categorical(['poly']),
        'svm__C': Real(1e-1, 1e3, prior='log-uniform'),
        'svm__gamma': Real(1e-4, 1e0, prior='log-uniform'),
        'svm__degree': Integer(2, 4)
    }
]

# Instantiate BayesSearchCV
bayes_search = BayesSearchCV(
    svm_pipeline_bayes,
    search_space,
    n_iter=50, # Number of parameter settings that are sampled
    cv=3,
    scoring='accuracy',
    random_state=42, # for reproducibility
    n_jobs=-1, # Use all available cores
    verbose=0 # Suppress verbose output
)

# Fit BayesSearchCV to the training data
bayes_search.fit(train_X, train_y)

# Print the best parameters and best score
print("Best parameters from BayesSearchCV:", bayes_search.best_params_)
print("Best cross-validation accuracy from BayesSearchCV: %.4f" % bayes_search.best_score_)

# Evaluate on test data
print(f'Test accuracy with best BayesSearchCV model: {bayes_search.best_estimator_.score(test_X, test_y):.2f}')


**Reasoning**:
The previous code failed because the `skopt` library was not found. I need to install `scikit-optimize` before attempting to import `BayesSearchCV` and other modules from it. A `!pip install` command should be in its own code cell.



In [32]:
!pip install scikit-optimize

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.8/107.8 kB 3.3 MB/s eta 0:00:00


**Reasoning**:
Now that `scikit-optimize` has been successfully installed, I will re-execute the `BayesSearchCV` code block to perform the hyperparameter tuning as requested in the subtask. This will allow the `skopt` modules to be imported correctly.



In [33]:
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

# Import BayesSearchCV and space objects
from skopt import BayesSearchCV
from skopt.space import Real, Integer, Categorical

# Create a new pipeline instance for BayesSearchCV
svm_pipeline_bayes = Pipeline([
    ("preprocessing", prep_pipeline),
    ("svm", SVC()),
])

# Define the parameter space for BayesSearchCV
# Note: BayesSearchCV takes a single dictionary for the search space, not a list of dictionaries like GridSearchCV/RandomizedSearchCV
# We need to explicitly define the categorical kernel and then specify parameters conditionally or define a broader space.
# For simplicity and to match the previous approach, we'll define a single space that covers all kernels,
# but only relevant parameters will be used based on the 'svm__kernel' choice by SVC internally.

search_space = [
    {
        'svm__kernel': Categorical(['linear']),
        'svm__C': Real(1e-1, 1e3, prior='log-uniform')
    },
    {
        'svm__kernel': Categorical(['rbf']),
        'svm__C': Real(1e-1, 1e3, prior='log-uniform'),
        'svm__gamma': Real(1e-4, 1e0, prior='log-uniform')
    },
    {
        'svm__kernel': Categorical(['poly']),
        'svm__C': Real(1e-1, 1e3, prior='log-uniform'),
        'svm__gamma': Real(1e-4, 1e0, prior='log-uniform'),
        'svm__degree': Integer(2, 4)
    }
]

# Instantiate BayesSearchCV
bayes_search = BayesSearchCV(
    svm_pipeline_bayes,
    search_space,
    n_iter=50, # Number of parameter settings that are sampled
    cv=3,
    scoring='accuracy',
    random_state=42, # for reproducibility
    n_jobs=-1, # Use all available cores
    verbose=0 # Suppress verbose output
)

# Fit BayesSearchCV to the training data
bayes_search.fit(train_X, train_y)

# Print the best parameters and best score
print("Best parameters from BayesSearchCV:", bayes_search.best_params_)
print("Best cross-validation accuracy from BayesSearchCV: %.4f" % bayes_search.best_score_)

# Evaluate on test data
print(f'Test accuracy with best BayesSearchCV model: {bayes_search.best_estimator_.score(test_X, test_y):.2f}')


KeyboardInterrupt: 